# Anomaly Detection — Finding Outliers in Data

- **Isolation Forest**: Random partitioning isolates anomalies faster
- **Local Outlier Factor (LOF)**: Density-based local anomalies
- **One-Class SVM**: Boundary-based outlier detection
- **Statistical Methods**: Z-score, IQR
- **Visualization** & evaluation on labeled data

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## 1. Create Dataset with Anomalies

In [ ]:
# Generate normal data + inject anomalies
np.random.seed(42)
n_normal, n_anomaly = 300, 30

# Normal data: two Gaussian clusters
X_normal, _ = make_blobs(n_samples=n_normal, centers=[[2, 2], [-2, -2]], 
                         cluster_std=0.8, random_state=42)

# Anomalies: scattered uniformly
X_anomaly = np.random.uniform(low=-6, high=6, size=(n_anomaly, 2))

X = np.vstack([X_normal, X_anomaly])
y_true = np.array([1] * n_normal + [-1] * n_anomaly)  # 1=normal, -1=anomaly

plt.figure(figsize=(8, 6))
plt.scatter(X_normal[:, 0], X_normal[:, 1], c='steelblue', s=20, label=f'Normal ({n_normal})')
plt.scatter(X_anomaly[:, 0], X_anomaly[:, 1], c='red', s=50, marker='x', label=f'Anomaly ({n_anomaly})')
plt.title('Dataset with Anomalies')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Isolation Forest

**Intuition**: Anomalies are few and different → they get isolated in fewer splits.

- Builds random trees that recursively split features at random values
- Anomalies have **shorter average path lengths** (fewer splits to isolate)
- `contamination`: expected proportion of outliers

In [ ]:
# Isolation Forest
iso_forest = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
iso_pred = iso_forest.fit_predict(X)
iso_scores = iso_forest.decision_function(X)

print('Isolation Forest Results:')
print(classification_report(y_true, iso_pred, target_names=['Anomaly', 'Normal']))

## 3. Local Outlier Factor (LOF)

**Intuition**: Compare local density of a point to its neighbors.  
Points in low-density regions relative to neighbors are outliers.

In [ ]:
# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
lof_pred = lof.fit_predict(X)
lof_scores = lof.negative_outlier_factor_

print('LOF Results:')
print(classification_report(y_true, lof_pred, target_names=['Anomaly', 'Normal']))

## 4. One-Class SVM

Learns a decision boundary around the normal data.

In [ ]:
# One-Class SVM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)
svm_pred = oc_svm.fit_predict(X_scaled)
svm_scores = oc_svm.decision_function(X_scaled)

print('One-Class SVM Results:')
print(classification_report(y_true, svm_pred, target_names=['Anomaly', 'Normal']))

## 5. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = [('Isolation Forest', iso_pred), ('LOF', lof_pred), ('One-Class SVM', svm_pred)]

for ax, (name, pred) in zip(axes, methods):
    normal_mask = pred == 1
    anomaly_mask = pred == -1
    ax.scatter(X[normal_mask, 0], X[normal_mask, 1], c='steelblue', s=20, alpha=0.5, label='Normal')
    ax.scatter(X[anomaly_mask, 0], X[anomaly_mask, 1], c='red', s=50, marker='x', label='Anomaly')
    
    # Mark misclassified
    misclassified = pred != y_true
    ax.scatter(X[misclassified, 0], X[misclassified, 1], s=100, facecolors='none',
               edgecolors='orange', linewidths=2, label='Misclassified')
    ax.set_title(name, fontsize=13)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Anomaly Detection Methods — Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Anomaly Score Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
scores = [('Isolation Forest', iso_scores), ('LOF', lof_scores), ('One-Class SVM', svm_scores)]

for ax, (name, score) in zip(axes, scores):
    ax.hist(score[y_true == 1], bins=30, alpha=0.6, label='Normal', color='steelblue')
    ax.hist(score[y_true == -1], bins=30, alpha=0.6, label='Anomaly', color='red')
    ax.set_title(f'{name} — Score Distribution')
    ax.set_xlabel('Anomaly Score'); ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. When to Use Which

| Method | Best For | Scalability |
|--------|----------|-------------|
| **Isolation Forest** | High-dim data, large datasets | Excellent |
| **LOF** | Local anomalies, varying densities | Moderate |
| **One-Class SVM** | Clean training data (no anomalies) | Low |
| **Z-Score/IQR** | Simple univariate outlier detection | Excellent |
| **Autoencoders** | Complex patterns, deep learning | High (GPU) |

### Real-World Applications:
- Fraud detection (credit cards, insurance)
- Network intrusion detection
- Manufacturing defect detection
- Medical anomaly (rare disease markers)